# スタッキング検証 ― LGBM(v10) に TCN(v12) 予測を特徴として入力

v10 の best 特徴セット(順方向+逆方向, `cmp_` 除外)に、TCN の OOF 予測 `tcn_pred` を
1列足して GroupKFold(5) by well で CV。`tcn_pred` あり/なしで overall RMSE を比較する。

リーク安全性: `tcn_pred = artifacts_v12/oof_tcn.npy` は各行が「その坑井を学習していないTCN」の
予測(坑井グループOOF)。よって特徴自体がリークせず、LGBMの val 指標は不偏。

参考: 加重ブレンド best = 7.3924(LGBM単独 7.5537)。スタッキングはこれ以上を狙う。
test提出時は TCN を全train再学習→test予測→特徴化→LGBM(全train再学習) の二段が必要。

メモリ: parquet を全列 DataFrame 化する(v10学習と同等)。20GB制約に注意。重ければ
feature を絞る or float32 化。

In [1]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import GroupKFold
from sklearn.metrics import root_mean_squared_error

PROJECT  = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
PARQUET  = PROJECT / "v10" / "artifacts_v10" / "data" / "train.parquet"
OOF_TCN  = PROJECT / "artifacts_v12" / "oof_tcn.npy"
N_SPLITS = 5
DEVICE_TYPE = "cpu"
BASE_MODE = "fwd_bwd"   # ベース特徴セット: "fwd" / "fwd_bwd" / "fwd_bwd_cmp"
NON_FEAT = {"well", "id", "target", "tcn_pred"}

LGB_PARAMS = dict(
    boosting_type="gbdt", num_leaves=255, min_child_samples=15,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=3.0, reg_alpha=0.05, objective="regression",
    verbose=-1, n_jobs=-1, max_bin=255,
    learning_rate=0.030, n_estimators=5000, seed=123,
)
EARLY_STOP = 200

In [2]:
# 読み込み + tcn_pred 付与
df = pd.read_parquet(PARQUET)
oof_tcn = np.load(OOF_TCN).astype(np.float32)
assert len(oof_tcn) == len(df), (len(oof_tcn), len(df))
df["tcn_pred"] = oof_tcn
groups = df["well"]
print(f"rows={len(df)}  wells={groups.nunique()}  cols={df.shape[1]}")

def base_features(cols, mode):
    feats = [c for c in cols if c not in NON_FEAT]
    if mode == "fwd":
        feats = [c for c in feats if not c.startswith("bpf_") and not c.startswith("cmp_")]
    elif mode == "fwd_bwd":
        feats = [c for c in feats if not c.startswith("cmp_")]
    return feats

base_feats = base_features(df.columns, BASE_MODE)
print(f"base({BASE_MODE}) features = {len(base_feats)}")

rows=3783989  wells=773  cols=659
base(fwd_bwd) features = 627


In [3]:
def run_cv(df, features, groups, n_splits, params, early_stop, device_type):
    X = df[features]; y = df["target"].to_numpy(np.float32)
    oof = np.full(len(df), np.nan, np.float32)
    gkf = GroupKFold(n_splits=n_splits)
    for k, (tr, va) in enumerate(gkf.split(X, y, groups)):
        pr = dict(params); pr["device_type"] = device_type
        m = LGBMRegressor(**pr)
        m.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])], eval_metric="rmse",
              callbacks=[early_stopping(early_stop, verbose=False), log_evaluation(0)])
        oof[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration_).astype(np.float32)
        print(f"    fold {k}: rmse={root_mean_squared_error(y[va], oof[va]):.4f}  it={m.best_iteration_}")
    overall = float(root_mean_squared_error(y, oof))
    tmp = pd.DataFrame({"well": groups.to_numpy(), "e2": (oof - y) ** 2})
    per_well = float(np.sqrt(tmp.groupby("well")["e2"].mean()).mean())
    return oof, overall, per_well

In [ ]:
# (A) ベースのみ  (B) ベース + tcn_pred
results = {}
for label, feats in [("base", base_feats), ("base + tcn_pred", base_feats + ["tcn_pred"])]:
    print(f"\n===== {label}  (features={len(feats)}) =====")
    t0 = time.perf_counter()
    oof, overall, per_well = run_cv(df, feats, groups, N_SPLITS, LGB_PARAMS, EARLY_STOP, DEVICE_TYPE)
    results[label] = (overall, per_well)
    print(f"  overall RMSE = {overall:.4f}  (坑井平均 {per_well:.4f}, {time.perf_counter()-t0:.0f}s)")

print("\n==== summary ====")
b = results["base"][0]
for label, (ov, pw) in results.items():
    print(f"{label:18s}: overall={ov:.4f}  per_well={pw:.4f}  Δ={ov-b:+.4f}")
print("Δ は base との overall 差。マイナスなら tcn_pred 特徴で改善(=スタッキング有効)。")


===== base  (features=627) =====
